# Validate dictionary declarations and their supported boundaries.

Run the setup first, then each step in order. This notebook executes the same declarations, calls and assertions as the [script](53_dictionary_catalogues.py). Outputs were captured by direct Python execution in a fresh process. No Jupyter kernel was used.


In [ ]:
!pip install -q aoa-action-machine

In [2]:
from typing import Annotated

from pydantic import Field

from aoa.action_machine.domain import AssociationOne, BaseCoreEntity, BaseEnumEntity, NoInverse, value
from aoa.action_machine.domain.base_domain import BaseDomain
from aoa.action_machine.domain.exceptions import EntityFieldDeclarationError
from aoa.action_machine.graph.core.exclude_graph_model import exclude_graph_model
from aoa.action_machine.graph.validators.entity_field_validator import validate_entity_fields
from aoa.action_machine.intents.entities import entities
from aoa.action_machine.intents.entity import entity
from aoa.action_machine.intents.meta import meta
from aoa.action_machine.resources.base_storage import BaseStorage


class Shop53Domain(BaseDomain):
    """Group the shop declarations."""

    name = "shop"
    description = "Shop model"


class EmptyEnumEntity(BaseEnumEntity):
    """Declare no values; ``@entity`` refuses an empty dictionary, so this one stays undecorated."""


@entity(description="Offer", domain=Shop53Domain)
class Offer53CoreEntity(BaseCoreEntity):
    """Type a field with a dictionary that declares nothing."""

    id: str = Field(description="Offer identifier")
    region: Annotated[AssociationOne[EmptyEnumEntity], NoInverse()] = Field(description="Primary region")


@meta(description="Stores the shop53 entities")
@entities(
    Offer53CoreEntity,
)
class Shop53Storage(BaseStorage):
    """Hold the shop53 entities."""


def case_53() -> None:
    """Run this one learning experiment."""
    try:
        validate_entity_fields([Offer53CoreEntity])
    except EntityFieldDeclarationError as error:
        print(type(error).__name__ + ":", error)
    else:
        raise AssertionError("Expected declaration refusal")


class Shop54Domain(BaseDomain):
    """Group the shop declarations."""

    name = "shop"
    description = "Shop model"


@exclude_graph_model
@entity(description="Regions kept out of the graph", domain=Shop54Domain)
class HiddenRegion54EnumEntity(BaseEnumEntity):
    """Stay out of the graph on purpose."""

    EU = value(code="eu", display_name="Europe")


@entity(description="Offer", domain=Shop54Domain)
class Offer54CoreEntity(BaseCoreEntity):
    """Reference a dictionary the graph does not carry."""

    id: str = Field(description="Offer identifier")
    region: Annotated[AssociationOne[HiddenRegion54EnumEntity], NoInverse()] = Field(description="Primary region")


@meta(description="Stores the shop54 entities")
@entities(
    HiddenRegion54EnumEntity,
    Offer54CoreEntity,
)
class Shop54Storage(BaseStorage):
    """Hold the shop54 entities."""


def case_54() -> None:
    """Run this one learning experiment."""
    try:
        validate_entity_fields([Offer54CoreEntity])
    except EntityFieldDeclarationError as error:
        print(type(error).__name__ + ":", error)
    else:
        raise AssertionError("Expected declaration refusal")

## Case 53


In [3]:
case_53()

EntityFieldDeclarationError: Field 'region' on entity 'Offer53CoreEntity' is invalid: EmptyEnumEntity declares no values; type the field with a concrete dictionary.


## Case 54


In [4]:
case_54()

EntityFieldDeclarationError: Field 'region' on entity 'Offer54CoreEntity' is invalid: HiddenRegion54EnumEntity has no node in the graph: it is not a declared @entity class, or it is marked exclude_graph_model, so the reference could not be drawn. Keep the dictionary in the graph.
